# E1.7 · Continuous control verification

**Function E — AI Governance for Agentic Systems → Building the Governance Framework — Risk and Control**  ·  *Security of AI*

Builds on **[E1.6 · Operating vs outcome guardrails](https://spbreed.github.io/cyber-commons/lessons/E1.6.html)**.

| | |
|---|---|
| Tools used | OPA, OSCAL, GLM-4.6, Claude Haiku 4.5 |

## What this lesson is

**What it covers.** Automate one evidence package on a schedule.

**Why a security engineer needs it.** Automating judgment instead of evidence collection. The control it builds is: agent-assisted evidence collection, drift detection, exception tracking.

This is a **control** lesson: it builds the mechanism, then breaks it, so you can see what the control is actually load-bearing for rather than taking the claim on trust.

## 1 · The hook

A control that is verified annually is a control you know about once a year. Continuous verification is the only version of assurance that keeps up with a system whose behaviour changes between tests.

> **At CyberTravels.** A control verified once a year on a system whose prompt changed on Tuesday. Continuous verification is the only version of assurance that keeps up with CyberTravels.

## 2 · The framework

```
   annual                          continuous
   +-------------+                 +-------------------------+
   | one sample  |                 | probe on every change   |
   | one date    |      vs         | sample continuously     |
   | one signature|                | escalate on failure     |
   +-------------+                 +-------------------------+

   assurance that keeps up with a system that changes weekly
```

Continuous control verification is the operating model that follows from E1.1.

The number that matters is not how much passed once. It is **how much is
currently evidenced** — controls whose most recent test is passing *and* within
its freshness window.

Three states, and the third is the one classical GRC tooling cannot express:

- **PASS** — tested, passing, in window.
- **FAIL** — tested, failing. Honest and actionable.
- **STALE** — tested, was passing, out of window. **Not a pass.**

Plus the absence state: no evidence at all, which is different from failing and
is often the largest category in a first assessment.

## 3 · Collecting the runtime evidence, as a skill

Automating a control means something has to go and look. For the network and logging controls that underwrite every default-deny claim CyberTravels makes, that is a posture collector: egress rules, private endpoints, route tables, key policies, and whether the audit trail is not merely enabled but **delivering**. It collects; it does not conclude. This is the file in this repository:

In [ ]:
# skills/attestation/aws-runtime-posture-collector/SKILL.md — embedded verbatim from the repository.
# This is the file itself, not a paraphrase of it.
SKILL_MD = r"""---
name: aws-runtime-posture-collector
description: >-
  Snapshot a deployment's cloud network, crypto and logging posture as
  evidence for default-deny and egress controls. Use to evidence network
  isolation, to check whether private endpoints and endpoint policies are in
  place, or to record the runtime configuration an attestation depends on.
allowed-tools: Bash, Read
---

# Aws Runtime Posture Collector

**Controls:** Controls 1 and 2 — runtime posture

## What this collects

Configuration state, at a point in time, for the network and crypto boundary
around one deployment. It is evidence for other skills' verdicts rather than a
verdict in itself.

## Procedure

1. **Security groups and network ACLs.** Enumerate egress rules. Any rule
   permitting `0.0.0.0/0` outbound is an egress-open finding regardless of what
   an application-layer policy says.
2. **Private endpoints.** Record whether a private endpoint exists for each
   model and gateway service in use, and read the endpoint policy — an
   unscoped endpoint policy is an endpoint that permits any principal.
3. **Route tables.** Identify NAT and internet gateways on the deployment's
   subnets. A private endpoint does not help if a default route to an internet
   gateway remains.
4. **Key policies.** Record key policies and condition keys that scope use to a
   specific service. Unconditioned key access is a finding.
5. **Logging.** Confirm the audit trail and log destinations are enabled and
   delivering, and record the retention.

## Output contract

```json
{
  "deployment_id": "str",
  "collected_at": "str",
  "network": {
    "egress_open_findings": [{"sg": "str", "rule": "str"}],
    "private_endpoints": [{"service": "str", "present": true, "policy_scoped": true}],
    "internet_route_present": false
  },
  "crypto": {"keys": [{"id": "str", "conditioned": true}]},
  "logging": {"audit_trail_enabled": true, "log_destinations": ["str"], "retention_days": 0},
  "verdict": "PASS|PARTIAL|FAIL"
}
```

## Failure modes

- **Reading configuration and calling it enforcement.** This skill records what
  is configured. Whether traffic actually obeys it is the egress verifier's job.
- **Ignoring the route table** because a private endpoint exists.
- **Recording that logging is enabled** without checking that it is delivering.
"""

In [ ]:
import json, re

def parse_skill(md):
    """Split a SKILL.md into (frontmatter dict, body).

    Frontmatter is a small, fixed subset of YAML: `key: value`, plus folded
    scalars (`description: >-`) whose continuation lines are indented. That is
    all a skill needs, and parsing it directly means no dependency.
    """
    if not md.startswith("---"):
        raise ValueError("a SKILL.md must open with a frontmatter block")
    _, front, body = md.split("---", 2)
    meta, key = {}, None
    for line in front.strip().splitlines():
        if not line.strip():
            continue
        if not line[0].isspace() and ":" in line:
            key, val = line.split(":", 1)
            key, val = key.strip(), val.strip()
            # `>-` and `|` open a folded block; the value is on the next lines
            meta[key] = "" if val in (">-", ">", "|", "|-") else val
        elif key is not None:
            meta[key] = (meta[key] + " " + line.strip()).strip()
    if "allowed-tools" in meta:
        meta["allowed-tools"] = [t.strip() for t in meta["allowed-tools"].split(",")
                                 if t.strip()]
    for required in ("name", "description"):
        if not meta.get(required):
            raise ValueError(f"skill is missing a {required!r}")
    return meta, body.strip()

_WORD = re.compile(r"[a-z][a-z-]{3,}")

def route(task, skills):
    """Pick the skill whose description best matches a task. Deterministic.

    The description is not documentation — it is the routing key. An agent
    decides whether to load a skill by reading it, so a vague description means
    the skill never fires when it should, and two overlapping descriptions mean
    the wrong one fires.

    Returns (pick, scores, margin). A margin of 0 means the top two scored the
    same and the "winner" is just whichever sorted first — an arbitrary answer
    wearing a confident face. Callers should refuse to auto-route on margin 0
    rather than pretend the tiebreak meant something.
    """
    want = set(_WORD.findall(task.lower()))
    def score(meta):
        return len(want & set(_WORD.findall(meta["description"].lower())))
    scores = {n: score(skills[n]) for n in sorted(skills)}
    # sort names first, then by score: ties must break identically on every
    # machine or the same task routes differently on two runs
    ranked = sorted(sorted(skills), key=lambda n: -scores[n])
    top = scores[ranked[0]]
    margin = top - (scores[ranked[1]] if len(ranked) > 1 else 0)
    return ranked[0], scores, margin

def contract_of(body):
    """The JSON block under '## Output contract' — the skill's machine promise."""
    # non-greedy across any prose between the heading and the fence
    m = re.search(r"## Output contract\b.*?```json\n(.*?)```", body, re.S)
    if not m:
        raise ValueError("skill declares no output contract")
    return json.loads(m.group(1))

def check(instance, contract, path="$"):
    """Structural conformance of an instance against a contract template.

    Returns the list of problems. An empty list means the shape is right — and
    that is *all* it means. Conformance is not accuracy: an empty findings list
    conforms perfectly and tells you nothing.
    """
    problems = []
    if isinstance(contract, dict):
        if not isinstance(instance, dict):
            return [f"{path}: expected an object, got {type(instance).__name__}"]
        for k, v in sorted(contract.items()):
            if k not in instance:
                problems.append(f"{path}.{k}: missing")
            else:
                problems += check(instance[k], v, f"{path}.{k}")
    elif isinstance(contract, list):
        if not isinstance(instance, list):
            return [f"{path}: expected a list, got {type(instance).__name__}"]
        for i, item in enumerate(instance):          # every element, same template
            problems += check(item, contract[0], f"{path}[{i}]")
    elif isinstance(contract, str) and "|" in contract:
        if instance not in contract.split("|"):
            problems.append(f"{path}: {instance!r} is not one of {contract}")
    elif isinstance(contract, bool):                  # before the numeric case:
        if not isinstance(instance, bool):            # bool is a subclass of int
            problems.append(f"{path}: expected bool, got {type(instance).__name__}")
    elif isinstance(contract, (int, float)):
        # JSON has one number type. A contract written `0` must accept 0.4, or
        # every cost and rate in the pipeline has to be rounded to satisfy a
        # checker rather than to be correct.
        if isinstance(instance, bool) or not isinstance(instance, (int, float)):
            problems.append(f"{path}: expected a number, got {type(instance).__name__}")
    elif not isinstance(instance, type(contract)):
        problems.append(f"{path}: expected {type(contract).__name__}, "
                        f"got {type(instance).__name__}")
    return problems

# Execute the skill above: parse skills/attestation/aws-runtime-posture-collector/SKILL.md into the two
# halves an agent uses — the frontmatter it routes on, and the body
# it follows.
meta, body = parse_skill(SKILL_MD)
print(f"loaded skill: {meta['name']}")
print(f"  tools it may use: {', '.join(meta.get('allowed-tools', [])) or '—'}")
print(f"  routing description: {len(meta['description'].split())} words")
print(f"  procedure: {len(body.splitlines())} lines")

## What you just proved

The skill loads and reports its shape, and the line to take from it is the boundary it draws: configuration is not enforcement. A private endpoint next to a route table with a NAT gateway is a recorded fact and an open path at the same time, and logging that is switched on but not delivering evidences nothing at all.

## Your turn

Automate the control with the shortest freshness window first — it is the one costing the most manual effort and going stale most often. One automated test converts an annual assertion into a live control.

---

**Next → [E1.8 · Third-party and model supply chain risk](https://spbreed.github.io/cyber-commons/lessons/E1.8.html)**

[All lessons](https://spbreed.github.io/cyber-commons/lessons/) · [This lesson's page](https://spbreed.github.io/cyber-commons/lessons/E1.7.html) · [Source](https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/labs/notebooks/E1.7.ipynb)

*Cyber Commons — a free, open commons for Cyber AI.*